In [ ]:

from trainer import *
from dataloader import *
from model import StrategicClassifierForWarmup, StrategicClassifierFiniteSet
import numpy as np
from model_utils import HingeLoss, BasicStrategicHingeLoss, AmbiguousStrategicHingeLoss
import torch
from datetime import datetime
import random
import matplotlib.pyplot as plt

In [ ]:
def generate_and_label_points(
    n_points=1000,
    x_low=-3.0,
    x_high=3.0,
    y_low=-3.0,
    y_high=3.0,
    shift=0.1,
    seed=101,
    cost_scaling = 1.0,
    device="cpu",
    dtype=torch.float32
):
    if seed is not None:
        np.random.seed(seed)

    W = [[1,0], [1,1], [1, -1]]
    b = [-1, 2, 2]
    W = np.asarray(W)
    b = np.asarray(b)

    w_chosen = W[0]
    b_chosen = b[0]

    X = np.column_stack([
        np.random.uniform(x_low, x_high, size=n_points),        # x ∈ [x_low, x_high]
        np.random.uniform(y_low, y_high, size=n_points) # y ∈ [y_low, y_high]
    ])

    margins = X @ W.T + b[None, :]

    cond_chosen = margins[:, 0] >= 0

    two_norm = (2.0 / cost_scaling) * np.linalg.norm(w_chosen)
    cond_intersection = np.all(margins >= -two_norm, axis=1)

    positive = cond_chosen | cond_intersection
    y = np.where(positive, 1, -1)

    X_moved = X.copy()
    X_moved[positive, 0] += shift / 2
    X_moved[~positive, 0] -= shift / 2

    X_final = X_moved
    y_final = y

    return X_final, y_final

In [ ]:
def plot_data(X, y, title="Training Data"):
    # X = X.detach().numpy()
    # y = y.detach().numpy()

    plt.figure(figsize=(6, 6))
    plt.scatter(X[y == -1][:, 0], X[y == -1][:, 1], c='red', label='Class -1', alpha=0.6)
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], c='blue', label='Class 1', alpha=0.6)
    plt.xlabel("x₁", fontsize=12)
    plt.ylabel("x₂", fontsize=12)
    plt.title(title, fontsize=14)
    plt.grid(True)
    plt.legend()
    plt.axis("equal")
    plt.show()

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(1)

In [ ]:
cost_scaling = 0.5
X,y = generate_and_label_points(n_points=1000, seed=101, shift=1.0, cost_scaling=cost_scaling, x_high=4.0, x_low=-6.0, y_high=10.0, y_low=-10.0)
plot_data(X, y)

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [ ]:
dataset = SCPIDataset(X, y)
dl_train, dl_val, dl_test = create_dataloaders(dataset, batch_size=1000, test_ratio=0.4, val_ratio=0.1, data_seed=100)

In [ ]:
model_3_classifiers = StrategicClassifierFiniteSet(d=2, num_classifiers=3, dev=0.4, cost_scaling=cost_scaling)
loss = AmbiguousStrategicHingeLoss()

opt = torch.optim.Adam(model_3_classifiers.parameters(), lr=0.006)

for name, p in model_3_classifiers.named_parameters():
    print(name, p)

trainer = StrategicTrainer(
    model=model_3_classifiers,
    loss_fn=loss,
    optimizer=opt,
    reg_classifier=0.001,
    reg_auxiliary=0.001,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
metrics = trainer.fit(
    dl_train=dl_train,
    dl_val=dl_val,
    num_epochs=500,
    early_stopping=None
)
trainer.predict(dl_test)

print(metrics)

In [ ]:
for name, p in model_3_classifiers.named_parameters():
    print(name, p)

print(metrics)
print(metrics["train_loss"])

In [ ]:
set_seed(1)
str_classification_model = StrategicClassifierForWarmup(d=2, cost_scaling=cost_scaling)
loss = BasicStrategicHingeLoss(scale_loss=cost_scaling)
opt = torch.optim.Adam(str_classification_model.parameters(), lr=0.006)
trainer = StrategicTrainer(
    model=str_classification_model,
    loss_fn=loss,
    optimizer=opt,
    reg_classifier=0.001,
    write_metrics=False,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)
trainer.fit(
    dl_train=dl_train,
    dl_val=dl_val,
    num_epochs=500,
    early_stopping=None,
    no_val = True
)
results = trainer.predict(dl_test)
print(results)

In [ ]:
print(results)
for name, param in str_classification_model.named_parameters():
    print(name, param.data)


In [ ]:
set_seed(1)
model_2_classifiers = StrategicClassifierFiniteSet(d=2, num_classifiers=2, dev=0.4, cost_scaling=cost_scaling)
loss = AmbiguousStrategicHingeLoss()

opt = torch.optim.Adam(model_2_classifiers.parameters(), lr=0.006)

for name, p in model_2_classifiers.named_parameters():
    print(name, p)

trainer = StrategicTrainer(
    model=model_2_classifiers,
    loss_fn=loss,
    optimizer=opt,
    reg_classifier=0.001,
    reg_auxiliary=0.001,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
metrics = trainer.fit(
    dl_train=dl_train,
    dl_val=dl_val,
    num_epochs=500,
    early_stopping=None
)
trainer.predict(dl_test)

print(metrics)


In [ ]:
print(metrics)
for name, param in model_2_classifiers.named_parameters():
    print(name, param.data)

In [ ]:
set_seed(1)
naive_model = StrategicClassifierForWarmup(d=2, cost_scaling=cost_scaling)
loss = HingeLoss()
opt = torch.optim.Adam(naive_model.parameters(), lr=0.006)
trainer = StrategicTrainer(
    model=naive_model,
    loss_fn=loss,
    optimizer=opt,
    reg_classifier=0.001,
    write_metrics=False,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
)
trainer.fit(
    dl_train=dl_train,
    dl_val=dl_val,
    num_epochs=500,
    early_stopping=None,
    no_val = True
)
results = trainer.predict(dl_test)
print(results)

In [ ]:
results
for name, param in naive_model.named_parameters():
    print(name, param.data)
print(results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D


def plot_model_classifiers(
    model,
    X,
    y,
    title="model decision boundaries",
    type="complex",
    alpha=1.0,
    show_effective_classifier=True,
    grid_resolution=300,
    chosen_line_resolution=1000
):
    # 1. Prepare Data
    X_np = X.detach().cpu().numpy()
    y_np = y.detach().cpu().numpy()

    plt.figure(figsize=(6, 4))

    # 2. Plot Data Points
    plt.scatter(
        X_np[y_np == -1][:, 0],
        X_np[y_np == -1][:, 1],
        c='red',
        label='Class -1',
        alpha=0.5,
        s=25,
        edgecolors='none'
    )

    plt.scatter(
        X_np[y_np == 1][:, 0],
        X_np[y_np == 1][:, 1],
        c='blue',
        label='Class 1',
        alpha=0.5,
        s=25,
        edgecolors='none'
    )

    # 3. Determine View Limits with Padding
    x_min, x_max = X_np[:, 0].min(), X_np[:, 0].max()
    y_min, y_max = X_np[:, 1].min(), X_np[:, 1].max()

    pad_x = (x_max - x_min) * 0.1
    pad_y = (y_max - y_min) * 0.1

    view_x_min, view_x_max = x_min - pad_x, x_max + pad_x
    view_y_min, view_y_max = y_min - pad_y, y_max + pad_y

    alpha_value = float(alpha.detach().cpu().item()) if torch.is_tensor(alpha) else float(alpha)

    # 4. Basic helper functions

    def tensor_to_numpy(w_param):
        return w_param.detach().cpu().numpy()

    def tensor_to_float(b_param):
        b_np = b_param.detach().cpu().numpy()
        return float(np.asarray(b_np).reshape(-1)[0])

    def get_line_coords(w, b):
        """
        Calculates the visible coordinates of the line:
            w[0] * x + w[1] * y + b = 0
        within the current view box.
        """
        if abs(w[0]) < 1e-5 and abs(w[1]) < 1e-5:
            return None, None

        borders = [
            ('bottom', view_y_min, 'horizontal'),
            ('top', view_y_max, 'horizontal'),
            ('left', view_x_min, 'vertical'),
            ('right', view_x_max, 'vertical')
        ]

        intersections = []

        for _, val, kind in borders:
            if kind == 'horizontal':
                if abs(w[0]) > 1e-5:
                    x_int = -(w[1] * val + b) / w[0]
                    if view_x_min <= x_int <= view_x_max:
                        intersections.append((x_int, val))
            else:
                if abs(w[1]) > 1e-5:
                    y_int = -(w[0] * val + b) / w[1]
                    if view_y_min <= y_int <= view_y_max:
                        intersections.append((val, y_int))

        intersections = list(set(intersections))

        if len(intersections) >= 2:
            intersections.sort(key=lambda p: p[0])
            return intersections[0], intersections[-1]

        return None, None

    def get_shift_amount(w_param):
        """
        Computes the dashed-line offset for a specific classifier v:
            (2 / alpha) * ||v||
        """
        v = tensor_to_numpy(w_param)
        return (2 / alpha_value) * np.linalg.norm(v)

    def plot_line_and_arrow(w_param, b_param, color, label, linewidth, is_main=False, linestyle='-', zorder=5):
        w = tensor_to_numpy(w_param)
        b = tensor_to_float(b_param)

        p1, p2 = get_line_coords(w, b)

        if p1 is not None and p2 is not None:
            plt.plot(
                [p1[0], p2[0]],
                [p1[1], p2[1]],
                color=color,
                linestyle=linestyle,
                label=label,
                alpha=1.0,
                linewidth=linewidth,
                zorder=zorder
            )

            # Only draw arrows for solid classifier lines
            if linestyle == '-':
                start_x = (p1[0] + p2[0]) / 2
                start_y = (p1[1] + p2[1]) / 2

                norm = np.linalg.norm(w)

                if norm > 1e-8:
                    w_norm = w / norm
                    scale = 1.0

                    plt.arrow(
                        start_x,
                        start_y,
                        w_norm[0] * scale,
                        w_norm[1] * scale,
                        head_width=0.3 if is_main else 0.25,
                        head_length=0.3 if is_main else 0.25,
                        fc=color,
                        ec=color,
                        zorder=zorder + 1,
                        length_includes_head=True
                    )

    # 5. AOP / Effective Classifier helper functions

    def clip_polygon_by_halfspace(polygon, w, b, eps=1e-9):
        """
        Clips a polygon by the halfspace:
            w[0] * x + w[1] * y + b >= 0
        """
        if len(polygon) == 0:
            return []

        clipped = []

        def inside(p):
            return np.dot(w, p) + b >= -eps

        def intersect(p1, p2):
            f1 = np.dot(w, p1) + b
            f2 = np.dot(w, p2) + b
            denom = f1 - f2

            if abs(denom) < 1e-12:
                return p1

            t = f1 / (f1 - f2)
            return p1 + t * (p2 - p1)

        prev = polygon[-1]
        prev_inside = inside(prev)

        for curr in polygon:
            curr_inside = inside(curr)

            if curr_inside:
                if not prev_inside:
                    clipped.append(intersect(prev, curr))
                clipped.append(curr)
            elif prev_inside:
                clipped.append(intersect(prev, curr))

            prev = curr
            prev_inside = curr_inside

        return clipped

    def compute_aop_polygon(classifiers, margin):
        """
        Computes a polygon approximation of the AOP:

            AOP = intersection over all classifiers of {x : w^T x + b >= 0}

        Since the AOP may be unbounded, we clip it inside a large box.
        """
        clip_x_min = view_x_min - margin
        clip_x_max = view_x_max + margin
        clip_y_min = view_y_min - margin
        clip_y_max = view_y_max + margin

        polygon = [
            np.array([clip_x_min, clip_y_min], dtype=float),
            np.array([clip_x_max, clip_y_min], dtype=float),
            np.array([clip_x_max, clip_y_max], dtype=float),
            np.array([clip_x_min, clip_y_max], dtype=float)
        ]

        for w, b in classifiers:
            polygon = clip_polygon_by_halfspace(polygon, w, b)

            if len(polygon) == 0:
                return []

        return polygon

    def point_segment_distance(points, a, b):
        """
        Distance from many 2D points to a line segment [a, b].
        points shape: (..., 2)
        """
        ab = b - a
        ab_norm_sq = np.dot(ab, ab)

        if ab_norm_sq < 1e-12:
            return np.linalg.norm(points - a, axis=-1)

        t = np.sum((points - a) * ab, axis=-1) / ab_norm_sq
        t = np.clip(t, 0.0, 1.0)

        projection = a + t[..., None] * ab
        return np.linalg.norm(points - projection, axis=-1)

    def distance_to_aop(points, classifiers, aop_polygon):
        """
        Distance from points to the AOP.

        If a point is inside all classifier-positive halfspaces, distance is 0.
        Otherwise, distance is approximated by distance to the clipped AOP polygon.
        """
        inside = np.ones(points.shape[:-1], dtype=bool)

        for w, b in classifiers:
            inside &= (points[..., 0] * w[0] + points[..., 1] * w[1] + b >= 0)

        dist = np.zeros(points.shape[:-1], dtype=float)

        if len(aop_polygon) == 0:
            dist[:] = np.nan
            return dist

        polygon = np.array(aop_polygon)
        min_dist = np.full(points.shape[:-1], np.inf)

        for i in range(len(polygon)):
            a = polygon[i]
            b = polygon[(i + 1) % len(polygon)]
            d = point_segment_distance(points, a, b)
            min_dist = np.minimum(min_dist, d)

        dist[~inside] = min_dist[~inside]

        return dist

    def plot_masked_polyline(points, mask, color='deeppink', linewidth=2.8, zorder=30):
        """
        Plots only the contiguous parts of a polyline where mask=True.
        """
        points = np.asarray(points)
        mask = np.asarray(mask)

        any_drawn = False
        start = None

        for i, keep in enumerate(mask):
            if keep and start is None:
                start = i

            if (not keep or i == len(mask) - 1) and start is not None:
                end = i if not keep else i + 1

                if end - start >= 2:
                    seg = points[start:end]
                    plt.plot(
                        seg[:, 0],
                        seg[:, 1],
                        color=color,
                        linewidth=linewidth,
                        linestyle='-',
                        zorder=zorder
                    )
                    any_drawn = True

                start = None

        return any_drawn

    def plot_effective_classifier(classifiers, chosen_w, chosen_b):
        """
        Effective classifier boundary of:

            H_chosen^+ union {x : dist(x, AOP) <= 2 / alpha}

        Therefore the boundary has two parts:

        1. The curve dist(x, AOP) = 2 / alpha,
           only where the chosen classifier is negative.

        2. The chosen classifier boundary,
           only where the negative side is not already covered by
           dist(x, AOP) <= 2 / alpha.

        In practice, part 2 is drawn on chosen_w^T x + chosen_b = 0
        only where dist(x, AOP) >= 2 / alpha.
        """
        radius = 2 / alpha_value

        span_x = view_x_max - view_x_min
        span_y = view_y_max - view_y_min
        margin = radius + max(span_x, span_y)

        aop_polygon = compute_aop_polygon(classifiers, margin=margin)

        if len(aop_polygon) == 0:
            print("AOP is empty in the plotted region, so the effective classifier was not drawn.")
            return False

        drew_something = False

        # ----------------------------------------------------
        # Part 1:
        # Draw dist(x, AOP) = 2 / alpha only where chosen is negative
        # ----------------------------------------------------
        xs = np.linspace(view_x_min, view_x_max, grid_resolution)
        ys = np.linspace(view_y_min, view_y_max, grid_resolution)

        XX, YY = np.meshgrid(xs, ys)
        grid_points = np.stack([XX, YY], axis=-1)

        D = distance_to_aop(grid_points, classifiers, aop_polygon)

        chosen_scores = chosen_w[0] * XX + chosen_w[1] * YY + chosen_b

        # Keep the AOP-distance curve only in the negative region of chosen
        D_masked = np.where(chosen_scores < 0, D, np.nan)

        if not np.all(np.isnan(D_masked)):
            d_min = np.nanmin(D_masked)
            d_max = np.nanmax(D_masked)

            if d_min <= radius <= d_max:
                plt.contour(
                    XX,
                    YY,
                    D_masked,
                    levels=[radius],
                    colors='deeppink',
                    linewidths=2.8,
                    linestyles='-',
                    zorder=30
                )
                drew_something = True

        # ----------------------------------------------------
        # Part 2:
        # Draw only the relevant parts of the chosen classifier
        # ----------------------------------------------------
        p1, p2 = get_line_coords(chosen_w, chosen_b)

        if p1 is not None and p2 is not None:
            p1 = np.array(p1, dtype=float)
            p2 = np.array(p2, dtype=float)

            t_vals = np.linspace(0.0, 1.0, chosen_line_resolution)
            chosen_line_points = p1[None, :] + t_vals[:, None] * (p2 - p1)[None, :]

            dist_on_chosen_line = distance_to_aop(
                chosen_line_points,
                classifiers,
                aop_polygon
            )

            # The chosen classifier is part of the effective boundary only
            # where the negative side is outside the AOP-radius region.
            tol = 1e-6
            chosen_effective_mask = dist_on_chosen_line >= radius - tol

            drew_chosen_segments = plot_masked_polyline(
                chosen_line_points,
                chosen_effective_mask,
                color='deeppink',
                linewidth=2.8,
                zorder=31
            )

            drew_something = drew_something or drew_chosen_segments

        return drew_something

    # 6. Draw Classifiers

    effective_classifier_drawn = False

    # Complex scenario: multiple classifiers
    if hasattr(model, 'w_chosen') and hasattr(model, 'classifiers_disguise'):

        title = f"(k={model.num_classifiers}) model decision boundary"

        classifiers_np = []

        # Add disguise classifiers
        for w_d, b_d in zip(model.classifiers_disguise, model.b_disguise):
            classifiers_np.append(
                (
                    tensor_to_numpy(w_d),
                    tensor_to_float(b_d)
                )
            )

        # Add chosen classifier
        classifiers_np.append(
            (
                tensor_to_numpy(model.w_chosen),
                tensor_to_float(model.b_chosen)
            )
        )

        # A. Plot possible disguise classifiers
        for i, (w_d, b_d) in enumerate(zip(model.classifiers_disguise, model.b_disguise)):
            lbl = 'possible' if i == 0 else None

            plot_line_and_arrow(
                w_d,
                b_d,
                color='green',
                label=lbl,
                linewidth=2.5,
                is_main=False,
                zorder=5
            )

            # Dashed line uses the norm of this specific classifier
            shift_amount = get_shift_amount(w_d)

            dash_lbl = 'decision boundary' if i == 0 else None

            plot_line_and_arrow(
                w_d,
                b_d + shift_amount,
                color='green',
                label=dash_lbl,
                linewidth=2.0,
                linestyle='--',
                is_main=False,
                zorder=4
            )

        # B. Plot realized chosen classifier
        plot_line_and_arrow(
            model.w_chosen,
            model.b_chosen,
            color='black',
            label='realized',
            linewidth=2.8,
            is_main=True,
            zorder=10
        )

        # Dashed line for realized classifier
        shift_amount = get_shift_amount(model.w_chosen)

        plot_line_and_arrow(
            model.w_chosen,
            model.b_chosen + shift_amount,
            color='black',
            label='decision boundary',
            linewidth=2.0,
            linestyle='--',
            is_main=True,
            zorder=9
        )

        # C. Plot effective classifier
        if show_effective_classifier:
            effective_classifier_drawn = plot_effective_classifier(
                classifiers=classifiers_np,
                chosen_w=tensor_to_numpy(model.w_chosen),
                chosen_b=tensor_to_float(model.b_chosen)
            )

    # Simple scenario: single classifier
    elif hasattr(model, 'w'):

        shift_amount = get_shift_amount(model.w)

        plot_line_and_arrow(
            model.w,
            model.b,
            color='black',
            label='classifier',
            linewidth=2.5,
            is_main=True
        )

        plot_line_and_arrow(
            model.w,
            model.b + shift_amount,
            color='black',
            label='decision boundary',
            linewidth=2.0,
            linestyle='--',
            is_main=True
        )

    # 7. Final Formatting
    plt.xlabel("x₁")
    plt.ylabel("x₂")
    plt.title(title)

    plt.xlim(view_x_min, view_x_max)
    plt.ylim(view_y_min, view_y_max)

    # --- Custom Legend Logic ---
    handles, labels = plt.gca().get_legend_handles_labels()

    label_dict = {}

    for h, l in zip(handles, labels):
        if l not in label_dict:
            label_dict[l] = []
        label_dict[l].append(h)

    final_handles = []
    final_labels = []

    for l, h_list in label_dict.items():
        if l == "decision boundary":
            final_handles.append(tuple(h_list))
            final_labels.append(l)
        else:
            final_handles.append(h_list[0])
            final_labels.append(l)

    if effective_classifier_drawn:
        final_handles.append(
            Line2D(
                [0],
                [0],
                color='deeppink',
                linewidth=2.8,
                linestyle='-'
            )
        )
        final_labels.append("effective classifier")

    plt.legend(
        final_handles,
        final_labels,
        loc='lower left',
        handler_map={tuple: HandlerTuple(ndivide=None)}
    )

    plt.grid(True, alpha=0.3)
    plt.gca().set_aspect('equal', adjustable='datalim')

    # plt.savefig(f'decision_boundary_{type}.eps', format='eps', bbox_inches='tight')
    # plt.savefig(f'decision_boundary_{type}.pdf', format='pdf', bbox_inches='tight')
    # plt.savefig(f'decision_boundary_{type}.png', format='png', dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
plot_model_classifiers(model_3_classifiers, X, y, title="3-classifiers model decision boundaries", type="3_classifiers", alpha=cost_scaling)
plot_model_classifiers(model_2_classifiers, X, y, title="2-classifiers model decision boundaries", type="2_classifiers", alpha=cost_scaling)
plot_model_classifiers(naive_model, X, y, title="Naive model decision boundary", type="naive", alpha=cost_scaling)
plot_model_classifiers(str_classification_model, X, y, title="Strategic classifier decision boundary", type="strategic", alpha=cost_scaling)